In [22]:
import pandas as pd
import fsspec
from zipfile import ZipFile
from io import BytesIO

def load_specific_csv_from_zip(url, filename):
    """Load one specific CSV from multi-file ZIP"""
    with fsspec.open(url, 'rb') as f:
        with ZipFile(BytesIO(f.read())) as z:
            return pd.read_csv(z.open(filename))

url = "gs://agntworks-data-dev/wheelsup/raw/AgntWorks.zip"

In [23]:
search_activity = load_specific_csv_from_zip(url, 'AgntWorks/search_activity.csv')
icao = "gs://agntworks-data-dev/wheelsup/processed/transformed_icao_metro.csv"

df = search_activity.copy()
icao_map = pd.read_csv(icao)

cluster_dict = dict(zip(icao_map['icao'], icao_map['metro']))

print(f'Loaded {len(df):,} searches')
print(f'Columns: {df.columns.tolist()}')
print(f'Sample routes:')
print(df['route'].head(5).tolist())

Loaded 765,682 searches
Columns: ['singleSearchRequestId', 'clientUserId', 'memberId', 'createDate', 'isCartAbandoned', 'requestingApp', 'route']
Sample routes:
['KMBO-KSAT-KMBO', 'KMBO-KSAT-KMBO', 'KDAL-KASE-KDAL', 'KLBE-KTEB-KBED-KLBE', 'LIMC-LEPA-LIMC']


In [24]:
# Split route into list, build consecutive (from, to) pairs
df['_airports'] = df['route'].str.split('-')
df['_legs']     = df['_airports'].apply(lambda x: list(zip(x[:-1], x[1:])))

# Explode: one row per leg
df_legs = df.explode('_legs').copy()
df_legs[['from_airport', 'to_airport']] = pd.DataFrame(
    df_legs['_legs'].tolist(), index=df_legs.index
)
df_legs = df_legs.drop(columns=['_airports', '_legs'])
df_legs = df_legs.reset_index(drop=True)

print(f'Original searches : {len(df):,}')
print(f'Expanded legs     : {len(df_legs):,}')
print(f'Avg legs/search   : {len(df_legs)/len(df):.2f}')

Original searches : 765,682
Expanded legs     : 1,183,154
Avg legs/search   : 1.55


In [25]:
df_legs['from_metro'] = df_legs['from_airport'].map(cluster_dict).fillna('OTHER')
df_legs['to_metro']   = df_legs['to_airport'].map(cluster_dict).fillna('OTHER')

unmapped_from = df_legs['from_metro'].eq('OTHER').sum()
unmapped_to   = df_legs['to_metro'].eq('OTHER').sum()
print(f'Unmapped from_airport: {unmapped_from:,} ({unmapped_from/len(df_legs)*100:.1f}%)')
print(f'Unmapped to_airport  : {unmapped_to:,} ({unmapped_to/len(df_legs)*100:.1f}%)')

# Time conversion — inline, no reload needed
df_legs['createDate_utc'] = pd.to_datetime(df_legs['createDate'], utc=True)
df_legs['createDate_et']  = df_legs['createDate_utc'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
df_legs['date_et'] = df_legs['createDate_et'].dt.date
df_legs['hour_et'] = df_legs['createDate_et'].dt.hour
df_legs['dow_et']  = df_legs['createDate_et'].dt.day_name()
print(f'\nSample ET: {df_legs["createDate_et"].iloc[0]}')


Unmapped from_airport: 76,273 (6.4%)
Unmapped to_airport  : 84,455 (7.1%)

Sample ET: 2025-03-29 20:31:25


In [26]:
# Sanity checks
print('=== Sample rows ===')
print(df_legs[['singleSearchRequestId','route','from_airport','to_airport','from_metro','to_metro']].head(10).to_string())

print('\n=== Top cluster pairs ===')
top_pairs = (
    df_legs.groupby(['from_metro','to_metro'])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='searches')
)
print(top_pairs.to_string())

print('\n=== Route length distribution ===')
route_len = df['route'].str.split('-').apply(len)
print(route_len.value_counts().sort_index().rename('count').rename_axis('airports_in_route'))

=== Sample rows ===
   singleSearchRequestId                route from_airport to_airport     from_metro       to_metro
0                7154439       KMBO-KSAT-KMBO         KMBO       KSAT      Nashville    San Antonio
1                7154439       KMBO-KSAT-KMBO         KSAT       KMBO    San Antonio      Nashville
2                7154443       KMBO-KSAT-KMBO         KMBO       KSAT      Nashville    San Antonio
3                7154443       KMBO-KSAT-KMBO         KSAT       KMBO    San Antonio      Nashville
4                7154446       KDAL-KASE-KDAL         KDAL       KASE         Dallas         Denver
5                7154446       KDAL-KASE-KDAL         KASE       KDAL         Denver         Dallas
6                7154450  KLBE-KTEB-KBED-KLBE         KLBE       KTEB     Pittsburgh  New York City
7                7154450  KLBE-KTEB-KBED-KLBE         KTEB       KBED  New York City         Boston
8                7154450  KLBE-KTEB-KBED-KLBE         KBED       KLBE         Bo

In [ ]:
df_legs.to_csv('gs://agntworks-data-dev/wheelsup/processed/search_activity_metro_mapping.csv', index=False)
print(f'Saved: gs://agntworks-data-dev/wheelsup/processed/search_activity_metro_mapping')
print(f'Rows : {len(df_legs):,}')
print(f'Cols : {df_legs.columns.tolist()}')